# Indic Dubbing — XTTS Synthesis Run

Runs a whole synthesis bundle and prints QC metrics.

Decoder settings come from `colab/xtts_worker.py`, which encodes the findings in
`colab/xtts.md`: greedy decoding for reproducible duration, and short reference
conditioning. Change them there, not here, so the settings stay recorded in the
result artifact.

**Run the cells in order.** Cells 5, 11 and 13 are checks that gate whether the
rest of the numbers mean anything; do not skip them.

In [ ]:
# The fixed worker lives on the `test` branch. `main` still passes
# temperature=0.7 with no greedy flag, which is the random-duration bug this
# run exists to confirm is gone, so checking out the right branch is not
# optional.
!git clone https://github.com/ayushk1233/indic-dub-pipeline.git 2>/dev/null || echo "already cloned"
%cd /content/indic-dub-pipeline
!git fetch --all --quiet && git checkout test && git pull --quiet
!git log --oneline -1
!pip install -q -r colab/requirements.txt

In [ ]:
from colab.preflight import PreflightValidator

print(PreflightValidator().run())

Upload the `tts_bundle.zip` produced locally by `BundleExporter`.

It is written to `artifacts/<job_id>/tts_bundle.zip` by the pipeline's export
stage.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()

BUNDLE = Path("/content/tts_bundle")
BUNDLE.mkdir(exist_ok=True)

with zipfile.ZipFile(next(iter(uploaded)), "r") as z:
    z.extractall(BUNDLE)

for p in sorted(BUNDLE.rglob("*")):
    print(p.relative_to(BUNDLE))

## Check the bundle before spending GPU time

Two things here decide whether the run is worth trusting.

The reference clip length matters because the last lines of `colab/xtts.md`
record that greedy decoding degrades with short reference audio, while
`CONDITIONING_PARAMS` caps conditioning at 10 seconds. That contradiction was
never resolved. Note the number this prints, because it is the variable to
change if the voice comes back wrong.

The parameters are printed from the worker's own constants rather than typed
here, so what you read is what will actually run.

In [ ]:
import json
import wave
from pathlib import Path

from colab.xtts_worker import CONDITIONING_PARAMS, INFERENCE_PARAMS, MODEL_ID

BUNDLE = Path("/content/tts_bundle")

manifest = json.loads((BUNDLE / "manifest.json").read_text())
request = json.loads((BUNDLE / "request" / "synthesis_request.json").read_text())

print(f"job           {manifest['metadata']['job_id']}")
print(f"bundle        v{manifest['metadata']['bundle_version']}")
print(f"language      {request['language']}")
print(f"segments      {len(request['segments'])}")
print(f"sample rate   {request['output_sample_rate']} Hz")

with wave.open(str(BUNDLE / "request" / "reference.wav")) as w:
    ref_s = w.getnframes() / w.getframerate()
    print(f"reference     {ref_s:.2f}s at {w.getframerate()} Hz")

print()
print(f"model         {MODEL_ID}")
print(f"conditioning  {CONDITIONING_PARAMS}")
print(f"inference     {INFERENCE_PARAMS}")

assert INFERENCE_PARAMS.get("do_sample") is False, (
    "do_sample must be False. With sampling on, every duration below is one "
    "draw from a random distribution and nothing in this run is measurable."
)
assert "temperature" not in INFERENCE_PARAMS, (
    "temperature is inert under greedy decoding and must not be passed."
)
print()
print("Decoder settings OK.")

if ref_s < 10:
    print(
        f"NOTE: reference is {ref_s:.1f}s, shorter than the 24s originally "
        "used. If cloning quality is poor, try a longer reference before "
        "changing any decoder setting."
    )

## Synthesize every segment

Writes `output/seg_NNNNN.wav` per segment plus `output/synthesis_result.json`,
re-saved after each segment so a timeout still leaves usable work.

This loop has not been run before. Expect per-segment progress lines; a failing
segment is logged to `logs/errors.log` and does not stop the run.

In [ ]:
from pathlib import Path

from colab.xtts_worker import XTTSWorker

worker = XTTSWorker(Path("/content/tts_bundle"))
result = worker.run()

done = sum(1 for s in result.segments if s.status == "done")
print(f"\n{done}/{len(result.segments)} segments synthesized")

## QC report

In [ ]:
from pathlib import Path

from src.eval.harness import build_report, render_report

report = build_report(
    job_dir=Path("/content/tts_bundle"),
    bundle_dir=Path("/content/tts_bundle"),
)

print(render_report(report))

Reading the synthesis table:

- `ratio` above 1.00 means the audio does not fit its slot.
- `tempo` is the time-stretch that would be needed; past about 1.25 it sounds rushed.
- `cps` is the delivered speaking rate. Measured natural Hindi is 10.8 characters
  per second, so far below that means the model padded rather than spoke.
- `sim` is cosine similarity to the reference speaker. Below 0.75 the clone drifted.
  A column of dashes means similarity never computed at all, which is a failure to
  investigate rather than a passing result.
- `tok` near the decoder ceiling means generation never terminated on its own.

## Realized pace

The one number every local decision was guessing at. Assembly can absorb a
segment that overruns by up to 1.25x and no more, so the mean here predicts
whether the dub will fit before you carry anything home.

`pace` is delivered duration divided by what a native speaker needs for the same
text at the measured rate for this language.

In [ ]:
from src.eval.translation_metrics import natural_cps

rate = natural_cps(request["language"])
texts = {s["segment_id"]: s["text"] for s in request["segments"]}

print(f"natural rate for {request['language']}: {rate:.2f} chars/sec\n")
print(f"{'seg':>4} {'chars':>6} {'natural':>8} {'actual':>8} {'pace':>6} {'tokens':>7} {'sim':>6}")

paces = []

for s in result.segments:
    if s.status != "done":
        print(f"{s.segment_id:>4}   {s.status}")
        continue

    chars = len(texts.get(s.segment_id, "").strip())
    natural = max(chars / rate, 1e-6)
    pace = s.duration / natural
    paces.append(pace)

    sim = "-" if s.speaker_similarity is None else f"{s.speaker_similarity:.3f}"
    print(
        f"{s.segment_id:>4} {chars:>6} {natural:>7.2f}s {s.duration:>7.2f}s "
        f"{pace:>5.2f}x {str(s.gpt_tokens or '-'):>7} {sim:>6}"
    )

if paces:
    print(f"\nmean pace {sum(paces)/len(paces):.2f}x   max {max(paces):.2f}x")
    print("Under ~1.25x the fitting cascade can absorb it. Above that it cannot.")

missing_sim = sum(1 for s in result.segments if s.speaker_similarity is None)
if missing_sim:
    print(
        f"\nWARNING: speaker similarity missing for {missing_sim} segments. "
        "The similarity helper swallows exceptions, so this looks the same as "
        "a clip being too short. Worth checking before trusting the clone."
    )

## Determinism check

This gates every other number in the notebook. Re-synthesize one segment twice;
with greedy decoding the sample counts must match exactly.

If they do not, decoding is still stochastic and nothing above is a
measurement. Check `INFERENCE_PARAMS` in `colab/xtts_worker.py` and confirm you
are on the `test` branch.

In [ ]:
first = worker.request.segments[0]

a = worker.synthesize_segment(first)
b = worker.synthesize_segment(first)

print(f"run 1: {a.duration:.4f}s, {a.num_samples} samples, {a.gpt_tokens} tokens")
print(f"run 2: {b.duration:.4f}s, {b.num_samples} samples, {b.gpt_tokens} tokens")
print()

if a.num_samples == b.num_samples:
    print("DETERMINISTIC — the numbers above are measurements.")
else:
    print(
        f"STILL SAMPLING — differed by {abs(a.num_samples - b.num_samples)} "
        "samples. Every duration in this notebook is one random draw. Fix "
        "INFERENCE_PARAMS before going further."
    )

## Listen

The one check no metric replaces. Judge two things separately: does it sound
like the reference speaker, and is the Hindi intelligible at the speed it is
delivered.

In [ ]:
from IPython.display import Audio, display

print("Reference speaker:")
display(Audio("/content/tts_bundle/request/reference.wav"))

for segment in result.segments:
    if segment.status != "done":
        continue
    text = texts.get(segment.segment_id, "")
    print(f"\nsegment {segment.segment_id}: {segment.duration:.2f}s — {text}")
    display(Audio(f"/content/tts_bundle/{segment.audio_path}"))

## Download results

Unzip this over `artifacts/<job_id>/tts_bundle/` locally, then resume:

    ./venv/bin/python -m src.cli --input test.mp4 --job-id <job_id> --from-stage import

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("/content/tts_bundle_out", "zip", "/content/tts_bundle")
files.download("/content/tts_bundle_out.zip")